# 01 - Exploración de Datos

Notebook de exploración inicial del corpus de reseñas en español.

## Objetivos
- Cargar el CSV de reseñas usando `src.data.loader.load_reviews`.
- Inspeccionar distribución de categorías, ratings y longitudes.
- Detectar problemas de calidad (nulos, duplicados, idioma).
- Probar el preprocesamiento básico.

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("No se encontró la raíz del repositorio")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import load_reviews
from src.data.preprocessing import clean_text, tokenize

## 1. Cargar dataset

In [ ]:
df = load_reviews(REPO_ROOT / "data/sample/reviews_sample.csv")
audit = {
    "filas": len(df),
    "review_id_unicos": df["review_id"].nunique(),
    "textos_vacios": int(df["text"].astype(str).str.strip().eq("").sum()),
    "duplicados_exactos": int(df.duplicated().sum()),
    "ratings_invalidos": int((~df["rating"].between(1, 5)).sum()),
}
display(df.head())
audit


## 2. Distribución de ratings y categorías

In [ ]:
print(df['rating'].value_counts().sort_index())
print()
print(df['product_category'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['rating'].value_counts().sort_index().plot.bar(ax=axes[0], title='Distribución de ratings')
df['product_category'].value_counts().plot.bar(ax=axes[1], title='Reseñas por categoría')
plt.tight_layout()
plt.show()

## 3. Longitudes de texto

In [ ]:
df['n_tokens'] = df['text'].str.split().str.len()
df['n_tokens'].describe()

## 4. Preprocesamiento: vista previa

Aplicamos limpieza y tokenización para verificar que el módulo `preprocessing` funciona como se espera.

In [ ]:
sample_text = df.iloc[0]['text']
print('Original :', sample_text)
print('Limpio   :', clean_text(sample_text))
print('Tokens   :', tokenize(clean_text(sample_text), remove_stopwords=True))

## Próximos pasos
- Notebook `02_baseline_clasico.ipynb`: lexicón + SVM.
- Validar etiquetado débil con anotaciones manuales sobre una muestra.